# Fabric Project Documentator - Instalador

Ejecuta este notebook una sola vez en el workspace donde quieres usar el documentador.

Requisitos:
- Un Lakehouse disponible en el workspace
- GitHub Fine-grained PAT con permisos Contents -> Read-only sobre el repositorio

In [ ]:
# CONFIGURACION -- edita antes de ejecutar

DEST_LAKEHOUSE = "MyLakehouse"    # cambia por tu lakehouse
GITHUB_TOKEN   = ""               # pega tu fine-grained PAT

# No cambiar
GITHUB_REPO   = "ecartolariesteves/200-Data-Engineering-Projects"
GITHUB_BRANCH = "main"
GITHUB_FOLDER = "202-fabric/202.3-Fabric-Documentator/config"
GITHUB_ASSETS = "202-fabric/202.3-Fabric-Documentator/assets"
INSTALL_PATH  = "/lakehouse/default/Files/docs/config"
ASSETS_PATH   = "/lakehouse/default/Files/docs/img"

if not GITHUB_TOKEN:
    raise ValueError("Configura GITHUB_TOKEN antes de continuar.")

print("OK Configuracion cargada")
print(f"   Lakehouse : {DEST_LAKEHOUSE}")
print(f"   Repo      : {GITHUB_REPO}")

In [ ]:
import requests, base64, os, re

CONFIG_FILES = [
    "CELL_4_functions.py",
    "CELL_4_additions_v33.py",
    "CELL_5_v33.py",
    "CELL_6a_style_helpers.py",
    "CELL_6b_v32.py",
    "CELL_6c_section2.py",
    "CELL_6d_v32.py",
    "CELL_6e_v33.py",
    "CELL_6f_additions_v33.py",
    "CELL_6f_consumption_v35.py",
]

ASSET_FILES = ["project_logo.jpg"]

HEADERS = {
    "Authorization":        f"Bearer {GITHUB_TOKEN}",
    "Accept":               "application/vnd.github.v3+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

API_BASE = f"https://api.github.com/repos/{GITHUB_REPO}/contents"

def clean_markdown_links(content_bytes):
    """
    GitHub a veces convierte referencias a archivos .py en enlaces markdown
    del tipo [filename.py](http://filename.py) al servir el contenido.
    Esta funcion los limpia dejando solo el nombre original.
    """
    text = content_bytes.decode("utf-8", errors="replace")
    text = re.sub(r'\[([^\]]+\.py)\]\(http[^\)]+\)', r'\1', text)
    return text.encode("utf-8")

def download_file(github_folder, filename, dest_folder):
    url  = f"{API_BASE}/{github_folder}/{filename}?ref={GITHUB_BRANCH}"
    dest = os.path.join(dest_folder, filename)
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        if resp.status_code == 200:
            content_b64 = resp.json().get("content", "")
            content     = base64.b64decode(content_b64)
            # Limpiar enlaces markdown que GitHub puede insertar en comentarios
            content     = clean_markdown_links(content)
            with open(dest, "wb") as f:
                f.write(content)
            return True, None
        else:
            msg = resp.json().get("message", f"HTTP {resp.status_code}")
            return False, msg
    except Exception as e:
        return False, str(e)

# Crear carpetas
os.makedirs(INSTALL_PATH, exist_ok=True)
os.makedirs(ASSETS_PATH,  exist_ok=True)
print("OK Carpetas creadas\n")

# Descargar config
print(f"Descargando {len(CONFIG_FILES)} archivos de configuracion...")
errors = []
for fname in CONFIG_FILES:
    ok, err = download_file(GITHUB_FOLDER, fname, INSTALL_PATH)
    if ok:
        size = os.path.getsize(os.path.join(INSTALL_PATH, fname)) / 1024
        print(f"  OK  {fname} ({size:.1f} KB)")
    else:
        print(f"  ERROR  {fname}: {err}")
        errors.append(fname)

# Descargar assets
print(f"\nDescargando assets...")
for fname in ASSET_FILES:
    variants = [fname, fname.replace(".jpg", " (1).jpg")]
    installed = False
    for variant in variants:
        ok, err = download_file(GITHUB_ASSETS, variant, ASSETS_PATH)
        if ok:
            src = os.path.join(ASSETS_PATH, variant)
            dst = os.path.join(ASSETS_PATH, fname)
            if src != dst and os.path.exists(src):
                os.rename(src, dst)
            print(f"  OK  {fname}")
            installed = True
            break
    if not installed:
        print(f"  AVISO  {fname} no encontrado -- sube el logo manualmente a {ASSETS_PATH}/")

print("\n" + "="*50)
if not errors:
    print("OK Instalacion completada correctamente")
else:
    print(f"AVISO Instalacion con {len(errors)} error(es):")
    for e in errors:
        print(f"   - {e}")

In [ ]:
import os

print("Verificando instalacion...\n")
all_ok = True
for fname in CONFIG_FILES:
    path = os.path.join(INSTALL_PATH, fname)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"  OK  {fname} ({size:.1f} KB)")
    else:
        print(f"  FALTA  {fname}")
        all_ok = False

print()
logo = os.path.join(ASSETS_PATH, "project_logo.jpg")
if os.path.exists(logo):
    print(f"  OK  project_logo.jpg ({os.path.getsize(logo)/1024:.1f} KB)")
else:
    print(f"  AVISO  project_logo.jpg no encontrado -- sube el logo manualmente")

print("\n" + "="*50)
if all_ok:
    print("OK Todo listo -- importa FabricProjectDocumentator.ipynb")
    print("   y configura la Celda 2 con los datos del proyecto.")
else:
    print("AVISO Algunos archivos no se instalaron -- vuelve a ejecutar.")

In [ ]:
# DESINSTALACION -- descomenta para activar

# import shutil
# shutil.rmtree(INSTALL_PATH, ignore_errors=True)
# shutil.rmtree(ASSETS_PATH,  ignore_errors=True)
# print("OK Archivos eliminados")

print("Desinstalacion no ejecutada (comentada por seguridad)")